# Module 01 — Lecture 1: GPU Architecture

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_01_cuda_fundamentals/01_gpu_architecture.ipynb)

---

## Why GPUs for Neuroscience?

The human brain contains roughly **86 billion neurons**, each receiving input from thousands of synapses, updating its state every millisecond. Even a simplified model of 10,000 neurons requires computing 10,000 differential equations simultaneously — 10 million times per second of simulated time.

A modern CPU can execute **one equation at a time** per core (maybe 16 cores with parallelism). A GPU can execute **thousands of equations simultaneously**. This is not a minor speedup — for large network simulations it is the difference between hours and seconds.

**Learning objectives for this lecture:**
- Understand why GPU architecture suits massively parallel computation
- Identify the key components: SMs, CUDA cores, warps
- Map the GPU memory hierarchy (global → L2 → L1/shared → registers)
- Appreciate where neuroscience simulations fit this architecture

## 1. CPU vs GPU: A Fundamental Design Philosophy

### CPU: Optimized for Serial, Low-Latency Tasks

```
┌─────────────────────────────────────────┐
│                  CPU                    │
│  ┌──────┐ ┌──────┐ ┌──────┐ ┌──────┐  │
│  │Core 0│ │Core 1│ │Core 2│ │Core 3│  │   4–64 powerful cores
│  └──────┘ └──────┘ └──────┘ └──────┘  │   Large cache (MB)
│  ┌─────────────────────────────────┐   │   Branch prediction
│  │           L3 Cache (~16 MB)     │   │   Out-of-order execution
│  └─────────────────────────────────┘   │
└─────────────────────────────────────────┘
```

CPUs are designed to run **complex, branchy, sequential** code fast.  
Each core has deep pipelines, large caches, and hardware for branch prediction.

### GPU: Optimized for Parallel, High-Throughput Tasks

```
┌──────────────────────────────────────────────────────┐
│                        GPU                           │
│  ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ┌────┐ ...     │
│  │ SM │ │ SM │ │ SM │ │ SM │ │ SM │ │ SM │          │   100s of SMs
│  └────┘ └────┘ └────┘ └────┘ └────┘ └────┘          │   Each SM: 128 CUDA cores
│  ┌────────────────────────────────────────────────┐  │
│  │                 L2 Cache (~40 MB)              │  │
│  └────────────────────────────────────────────────┘  │
│  ┌────────────────────────────────────────────────┐  │
│  │              Global Memory (80 GB)             │  │
│  └────────────────────────────────────────────────┘  │
└──────────────────────────────────────────────────────┘
```

GPUs are designed to run **the same simple operation** on **massive datasets** simultaneously.

| Feature | CPU (modern) | GPU (A100) |
|---------|-------------|------------|
| Cores | 64 | 6,912 CUDA cores |
| Clock speed | 3–5 GHz | 1.4 GHz |
| Memory BW | ~50 GB/s | 2,039 GB/s |
| FP32 TFLOPS | ~1.5 | 312 |
| Cache per core | Large | Small |

> **Key insight:** GPUs trade per-core sophistication for massive parallelism. This is exactly what simulating thousands of identical neurons requires.

## 2. Inside a GPU: Streaming Multiprocessors

The fundamental compute unit of a GPU is the **Streaming Multiprocessor (SM)**.

```
┌─────────────────────── Streaming Multiprocessor ───────────────────────┐
│                                                                         │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐  ┌──────────┐               │
│  │ CUDA Core│  │ CUDA Core│  │ CUDA Core│  │ CUDA Core│ × 128         │
│  │  (FP32)  │  │  (FP32)  │  │  (FP32)  │  │  (FP32)  │               │
│  └──────────┘  └──────────┘  └──────────┘  └──────────┘               │
│                                                                         │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │             Shared Memory / L1 Cache  (~48–164 KB)              │   │
│  └─────────────────────────────────────────────────────────────────┘   │
│                                                                         │
│  ┌───────────────────┐   ┌───────────────────┐                         │
│  │  Register File    │   │   Warp Schedulers  │                         │
│  │  (65536 × 32-bit) │   │   (4 schedulers)   │                         │
│  └───────────────────┘   └───────────────────┘                         │
└─────────────────────────────────────────────────────────────────────────┘
```

A single GPU (e.g., RTX 4090) has **128 SMs**, each with 128 CUDA cores = **16,384 cores total**.

### The Warp: The GPU's Atomic Unit of Execution

Threads are grouped into **warps** of **32 threads**. All 32 threads in a warp execute the **same instruction simultaneously** (SIMT: Single Instruction, Multiple Threads).

```
Block of 256 threads:
  Warp 0:  threads  0-31   → execute instruction A together
  Warp 1:  threads 32-63   → execute instruction A together
  Warp 2:  threads 64-95   → execute instruction A together
  ...
  Warp 7:  threads 224-255 → execute instruction A together
```

**Warp divergence:** If threads in the same warp take different code paths (e.g., `if (neuron_type == EXCITATORY)`), the warp executes **both branches serially**, masking threads not on each branch. This halves throughput. Design kernels so threads in a warp take the same path.

> **Neuroscience implication:** When simulating a network with mixed excitatory/inhibitory neurons, separate them into different thread blocks to avoid warp divergence.

## 3. GPU Memory Hierarchy

Memory access speed is often the performance bottleneck. Understanding this hierarchy is critical.

```
  Speed      Memory Type          Scope         Size
  ─────      ────────────         ─────         ────
  FASTEST ▲  Registers            per thread    ~255 per thread
           │  Shared Memory (L1)   per block     48–164 KB per SM
           │  L2 Cache             whole GPU     40–80 MB
  SLOWEST ▼  Global Memory (DRAM)  whole GPU     8–80 GB

  Latency:  ~1 cycle  →  ~20 cycles  →  ~200 cycles  →  ~600 cycles
```

### Mapping to a Neuron Simulation

| Neuroscience Data | Best Memory Type | Why |
|-------------------|-----------------|-----|
| Neuron membrane voltage V[i] | Global memory | One per neuron, all threads need it |
| Gating variables m, h, n | Global memory | Updated each timestep |
| Synaptic weight matrix W | Global memory | Too large for shared |
| Timestep Δt, reversal potentials | Constant memory | Same for all neurons, cached |
| Intermediate ODE sub-steps | Registers | Private to each thread |
| Input currents within a block | Shared memory | Threads in block collaborate |

> **Key rule:** Minimize global memory accesses. Every read from global memory costs ~600 cycles. Loading a value once into shared memory and reusing it many times is the central optimization technique.

In [ ]:
# First, verify we have a GPU in this Colab session
!nvidia-smi

In [ ]:
# Query detailed GPU properties using a small CUDA program
%%writefile gpu_info.cu
#include <stdio.h>
#include <cuda_runtime.h>

int main() {
    int n_devices;
    cudaGetDeviceCount(&n_devices);
    printf("Number of GPUs: %d\n\n", n_devices);

    for (int d = 0; d < n_devices; d++) {
        cudaDeviceProp p;
        cudaGetDeviceProperties(&p, d);

        printf("=== GPU %d: %s ===\n", d, p.name);
        printf("  Compute capability:        %d.%d\n", p.major, p.minor);
        printf("  Streaming Multiprocessors: %d\n", p.multiProcessorCount);
        printf("  CUDA cores (est.):         %d\n", p.multiProcessorCount * 128);
        printf("  Warp size:                 %d\n", p.warpSize);
        printf("  Max threads per block:     %d\n", p.maxThreadsPerBlock);
        printf("  Max blocks per SM:         %d\n", p.maxBlocksPerMultiProcessor);
        printf("  Global memory:             %.1f GB\n", p.totalGlobalMem / 1e9);
        printf("  Shared mem per SM:         %zu KB\n", p.sharedMemPerMultiprocessor / 1024);
        printf("  L2 cache:                  %d MB\n", p.l2CacheSize / 1024 / 1024);
        printf("  Memory bandwidth:          %.0f GB/s\n",
               2.0 * p.memoryClockRate * (p.memoryBusWidth / 8) / 1e6);
        printf("\n");
    }
    return 0;
}

In [ ]:
!nvcc -o gpu_info gpu_info.cu && ./gpu_info

## 4. Connecting GPU Architecture to Neuroscience Simulations

Let's make this concrete. Suppose you want to simulate **N = 10,000 LIF neurons** for T = 1 second of biological time with Δt = 0.1 ms (10,000 time steps).

**Total operations:** 10,000 neurons × 10,000 steps = **100 million neuron updates**

### CPU Approach (serial)
```
for step in range(10000):         # 10,000 time steps
    for i in range(10000):        # 10,000 neurons
        update_neuron(i)          # ~10 FLOPs each
```
→ ~1 billion FLOPs / (10 GFLOPS per core) ≈ **100 seconds** on one CPU core

### GPU Approach (parallel)
```
for step in range(10000):         # still 10,000 time steps (sequential)
    update_all_neurons<<<...>>>() # ALL 10,000 neurons in parallel
```
→ Each step takes ~1 μs, total ≈ **10 ms** on a GPU

**Speedup: ~10,000×**

The key insight: **the time dimension is sequential** (neuron at t+1 depends on t), but **the neuron dimension is embarrassingly parallel** (neuron 0 at step t doesn't depend on neuron 1 at step t, assuming no instantaneous coupling). The GPU exploits this perfectly.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

# Visualize: CPU vs GPU time as a function of neuron count
# (theoretical model based on hardware specs)

N_values = np.logspace(1, 6, 50).astype(int)   # 10 to 1,000,000 neurons
T_steps  = 10000
flops_per_update = 15   # rough LIF kernel FLOPs

cpu_gflops = 50.0   # typical single-core FP32 throughput
gpu_gflops = 10000.0  # GPU (T4 = 8.1 TFLOPS, use conservative 10 TFLOPS)

cpu_time_s = (N_values * T_steps * flops_per_update) / (cpu_gflops * 1e9)
gpu_time_s = (N_values * T_steps * flops_per_update) / (gpu_gflops * 1e9)
# GPU has launch overhead floor ~1ms per step
gpu_time_s = np.maximum(gpu_time_s, T_steps * 1e-6)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.loglog(N_values, cpu_time_s, 'b-', linewidth=2, label='CPU (1 core)')
ax1.loglog(N_values, gpu_time_s, 'r-', linewidth=2, label='GPU (T4)')
ax1.axvline(10000, color='gray', linestyle='--', alpha=0.6, label='N=10,000 neurons')
ax1.set_xlabel('Number of neurons (N)', fontsize=13)
ax1.set_ylabel('Wall-clock time (seconds)', fontsize=13)
ax1.set_title('Simulation Time: CPU vs GPU', fontsize=14)
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

speedup = cpu_time_s / gpu_time_s
ax2.semilogx(N_values, speedup, 'g-', linewidth=2)
ax2.axvline(10000, color='gray', linestyle='--', alpha=0.6, label='N=10,000 neurons')
ax2.set_xlabel('Number of neurons (N)', fontsize=13)
ax2.set_ylabel('GPU Speedup (×)', fontsize=13)
ax2.set_title('GPU Speedup over Single CPU Core', fontsize=14)
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cpu_vs_gpu.png', dpi=150, bbox_inches='tight')
plt.show()

# Print key values
idx = np.searchsorted(N_values, 10000)
print(f"At N=10,000 neurons:")
print(f"  CPU time: {cpu_time_s[idx]:.1f} s")
print(f"  GPU time: {gpu_time_s[idx]*1000:.1f} ms")
print(f"  Speedup:  {speedup[idx]:.0f}×")

## 5. The CUDA Execution Model — Preview

Before writing code, here is the big picture of how CUDA organizes computation:

```
Host (CPU)                          Device (GPU)
──────────────────                  ─────────────────────────────────────
                                    Grid (entire kernel launch)
kernel<<<grid, block>>>()  ──────▶  ┌─────────────────────────────────┐
                                    │  Block (0,0)  │  Block (1,0)  │ …│
                                    │  ┌─────────┐  │  ┌─────────┐  │  │
                                    │  │Thread 0 │  │  │Thread 0 │  │  │
                                    │  │Thread 1 │  │  │Thread 1 │  │  │
                                    │  │  ...    │  │  │  ...    │  │  │
                                    │  │Thread N │  │  │Thread N │  │  │
                                    │  └─────────┘  │  └─────────┘  │  │
                                    └─────────────────────────────────┘
```

We will build this up in detail in Lecture 2. The key things to remember now:

1. **Grid** = the entire launch (all blocks)
2. **Block** = a group of threads that share fast shared memory and can synchronize
3. **Thread** = the individual worker — runs your kernel function

In neuroscience: **one thread = one neuron**.

## Summary

| Concept | Key Point |
|---------|----------|
| GPU vs CPU | GPU trades per-core power for massive parallelism |
| Streaming Multiprocessor | The basic GPU compute unit; contains CUDA cores + shared memory |
| Warp | 32 threads that execute in lockstep (SIMT) |
| Memory hierarchy | Registers → Shared → L2 → Global (fastest to slowest) |
| Neuroscience fit | Neurons are independent at each timestep → perfect GPU parallelism |

**Next lecture:** We will learn the CUDA programming model — how to write kernels, launch them, and index into arrays.

---

## Self-Check Questions

1. A warp has 32 threads. If an SM can run 48 warps simultaneously, how many threads are active per SM?
2. Why does warp divergence hurt performance? Give a neuroscience example where it might occur.
3. You have a simulation with N=65,536 neurons. Each neuron needs 5 float state variables. How much global memory do you need?
4. Why can't we use shared memory for the full synaptic weight matrix of 10,000×10,000 neurons?

---

**Next →** [02 — CUDA Programming Model](02_cuda_programming_model.ipynb) &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/praveen-dedigamage/Cuda-For-Computational-Neuroscience/blob/main/module_01_cuda_fundamentals/02_cuda_programming_model.ipynb)